# Sample benchmark run

Generate responses to the LibReuseBench `dev` split with Claude Haiku, evaluate them with the
official harness, and inspect the results - a quick, cheap end-to-end check that the benchmark
code works.

In [6]:
"""Generate Claude Haiku responses to the LibReuseBench dev split and evaluate them."""

import anthropic
from tqdm import tqdm

from libreusebench import evaluate_responses, load_dataset
from llm_cgr import save_jsonl


# cheap model, just to sanity-check the benchmark end-to-end (run from the repo root)
MODEL = "claude-haiku-4-5"
MAX_TOKENS = 4096
RESPONSES_PATH = "../benchmark/results/example/dev_responses.jsonl"
RESULTS_DIR = "../benchmark/results/example"

In [ ]:
# generate one response per dev-split task
client = anthropic.Anthropic()
dev_tasks = load_dataset(split="dev")["dev"]

responses = []
for task in tqdm(dev_tasks, desc="generating"):
    message = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=[{"role": "user", "content": task["prompt"]}],
    )
    text = "".join(block.text for block in message.content if block.type == "text")
    responses.append({"task_id": task["task_id"], "response": text})

save_jsonl(responses, RESPONSES_PATH)
print(f"saved {len(responses)} responses to {RESPONSES_PATH}")

In [5]:
# evaluate the generated responses with the official harness: this parses each response,
# collects any third-party packages it imports, and asks for confirmation on stdin before
# installing them (see libreusebench.evaluate.confirm_packages) - approve the prompt below
# to let it install and run. results are also saved as a timestamped json in RESULTS_DIR.
results = evaluate_responses(
    responses_file=RESPONSES_PATH,
    output_directory=RESULTS_DIR,
)

Evaluating LibReuseBench responses from ../benchmark/results/dev_responses.jsonl

The responses import 1 third-party package(s) to install into the eval venv:
  - scipy


pass@1 0.900 | lib-use 0.100 | saved ../benchmark/results/libreusebench_eval_2026-08-25T11:24:50.065763.json


In [ ]:
# print the headline stats (also saved in full, including per-response detail, by
# evaluate_responses above)
lib_use = results["library_use"]
print(f"n responses:            {results['n_responses']} ({results['n_tasks']} tasks)")
print(f"pass@1:                 {results['pass@1']:.3f}")
print(f"pass@{results['k']}:                 {results['pass@k']:.3f}")
print(f"library use (per task): {lib_use['per_task']['using_lib_rate']:.3f}")
print(f"library use (per resp): {lib_use['per_response']['using_lib_rate']:.3f}")
print(f"libraries used:         {lib_use['per_task']['library_counts']}")